In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# Define the symbol for the S&P500 index
symbol = "GOOG"

# Define the expiration date for the options
expiration_date = "2023-03-31"

# Get the options data for the symbol and expiration date from Yahoo Finance
options = yf.Ticker(symbol).option_chain(expiration_date)

# Extract the relevant information from the options data
strikes = options.calls.strike.values
implied_vols = options.calls.impliedVolatility.values

# Sort the data by strike price
strikes, implied_vols = zip(*sorted(zip(strikes, implied_vols)))

# Interpolate the implied volatilities to get a smooth curve
f = interp1d(strikes, implied_vols, kind="cubic")
x_interp = np.linspace(min(strikes), max(strikes), num=100)
y_interp = f(x_interp)

# Plot the volatility smile
plt.plot(strikes, implied_vols, "o", label="Implied volatilities")
plt.plot(x_interp, y_interp, label="Interpolated curve")
plt.xlabel("Strike price")
plt.ylabel("Implied volatility")
plt.title("Volatility smile for "+symbol+"\n Expiration="+expiration_date)
plt.legend()
plt.show()

In [ ]:
# Extract the call and put option data
calls = options.calls
puts = options.puts

In [ ]:
import numpy as np
import scipy.stats as si
from scipy.stats import norm
from scipy.optimize import brentq

def black_scholes_call(S, K, T, r, sigma):
    #Calculate the price of a European call option using the Black-Scholes formula.
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

def implied_volatility(S, K, T, r, C):
    #Calculate the implied volatility of an option using the Black-Scholes formula.
    def f(sigma):
        d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
        d2 = d1 - sigma*np.sqrt(T)
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2) - C
    try:
        return brentq(f, 1e-6, 1)
    except ValueError:
        return np.nan
    

In [ ]:
# Get the current stock price
stock = yf.Ticker(symbol)
current_price = stock.history().iloc[-1]["Close"]

# Define a range of strike prices
strike_range = np.linspace(current_price * 0.5, current_price * 1.5, num=100)

# Compute the implied volatilities for the range of strike prices
implied_vols = []
for strike in strike_range:
    option = stock.option_chain(expiration_date).calls.loc[stock.option_chain(expiration_date).calls["strike"]==strike]
    if not option.empty:
        bid_price = option.iloc[0]["bid"]
        ask_price = option.iloc[0]["ask"]
        if not np.isnan(bid_price) and not np.isnan(ask_price):
            C = (bid_price + ask_price) / 2.0
            implied_vol = implied_volatility(current_price, strike, (pd.to_datetime(expiration_date) - pd.to_datetime('today')).days/365, 0.01, C)
            implied_vols.append(implied_vol)
        else:
            implied_vols.append(np.nan)
    else:
        implied_vols.append(np.nan)

# Compute the weighted average of the implied volatilities
weights = [(T/365)**2 for T in range(1, len(strike_range)+1)]
weighted_implied_vols = [w * v for w, v in zip(weights, implied_vols) if not np.isnan(v)]
vix = 100 * np.sqrt(sum(weighted_implied_vols) * (365 / len(strike_range)) / sum(weights))

# Print the VIX value
print(f"VIX for {symbol}: {vix:.2f}")

In [ ]:
import yfinance as yf
import numpy as np
from scipy.interpolate import interp1d
import pandas as pd

# Define the symbol for the stock
symbol = "GOOG"

# Define the expiration date for the options
expiration_date = "2023-03-31"

# Get the options data for the symbol and expiration date from Yahoo Finance
options = yf.Ticker(symbol).option_chain(expiration_date)

# Get the spot price of the stock
stock = yf.Ticker(symbol)
spot_price = stock.history().iloc[-1]["Close"]

# Get the strike prices and option prices
strike_prices = options.calls['strike'].to_numpy()
option_prices = options.calls['lastPrice'].to_numpy()

# Calculate the option maturities in years
maturities = np.zeros(len(strike_prices))
for i in range(len(strike_prices)):
    last_trade_date = pd.to_datetime(options.calls['lastTradeDate'][i]).tz_localize(None)
    maturities[i] = (pd.to_datetime(expiration_date) - last_trade_date).days / 365.0

# Calculate the option implied volatilities
implied_vols = np.zeros(len(strike_prices))
for i in range(len(strike_prices)):
    implied_vols[i] = yf.Ticker(symbol).option_chain(expiration_date).calls.loc[yf.Ticker(symbol).option_chain(expiration_date).calls['strike'] == strike_prices[i]]['impliedVolatility'].values[0]

# Find the at-the-money (ATM) option
ATM_index = np.argmin(np.abs(strike_prices - spot_price))

# Define the range of strike prices to use for the VIX calculation
strikes_range = np.linspace(0.5 * spot_price, 1.5 * spot_price, num=50)

# Interpolate the option prices and implied volatilities
option_price_interp = np.interp(strikes_range, strike_prices, option_prices)
implied_vol_interp = np.interp(strikes_range, strike_prices, implied_vols)

# Calculate the variance of the option prices
variance = 0
for i in range(len(strikes_range)):
    strike = strikes_range[i]
    if strike < strike_prices[0] or strike > strike_prices[-1]:
        continue
    option_price = option_price_interp[i]
    implied_vol = implied_vol_interp[i]
    if i == ATM_index:
        weight = 1.0
    else:
        weight = (strike / spot_price) ** 2 * np.exp(0.25 * implied_vol ** 2 * maturities[ATM_index] - 0.25 * implied_vol ** 2 * maturities[i])
    variance += weight * ((option_price - option_price_interp[ATM_index]) / spot_price) ** 2 / maturities[i]

# Calculate the VIX
VIX = 100 * np.sqrt(variance * 365)
print("VIX for {} on {}: {:.2f}".format(symbol, expiration_date, VIX))

In [ ]:
import yfinance as yf
import numpy as np
from scipy.interpolate import interp1d

# Define the symbol for the S&P500 index
symbol = "SPY"

# Define the range of option expiration dates to consider
expirations = ["2023-04-21", "2023-05-19", "2023-06-16", "2023-07-21", "2023-09-15"]

# Retrieve the option chain data from Yahoo Finance
options = yf.Ticker(symbol).option_chain(expirations[0])

# Extract the strike prices and option prices
strike_prices = np.array(options.calls["strike"])
option_prices = np.array(options.calls["lastPrice"])

# Calculate the maturities of the options
maturities = np.zeros(len(strike_prices))
for i in range(len(strike_prices)):
    maturities[i] = (np.datetime64(expirations[0]) - np.datetime64(options.calls["lastTradeDate"][i])).astype('timedelta64[D]') / np.timedelta64(1, 'D') / 365.0

# Calculate the implied volatilities of the options
implied_vols = np.zeros(len(strike_prices))
for i in range(len(strike_prices)):
    # Use Black-Scholes model to calculate implied volatility
    # (code for Black-Scholes model omitted for brevity)
    implied_vols[i] = implied_volatility(option_prices[i], strike_prices[i], maturities[i], spot_price)#, risk_free_rate, dividend_yield)

# Interpolate the option prices and implied volatilities
option_price_interp = interp1d(strike_prices, option_prices, kind="linear", bounds_error=False, fill_value=(option_prices[0], option_prices[-1]))
implied_vol_interp = interp1d(strike_prices, implied_vols, kind="linear", bounds_error=False, fill_value=(implied_vols[0], implied_vols[-1]))

# Define the range of standardized strike prices to use
Ks = np.linspace(0.5 * strike_prices[0], 1.5 * strike_prices[-1], num=101)

# Calculate the corresponding option prices and implied volatilities for the standardized strike prices
option_prices_std = option_price_interp(Ks)
implied_vols_std = implied_vol_interp(Ks)

# Calculate the variance of the option prices
variance = 0
for i in range(len(strikes_range)):
    strike = strikes_range[i]
    if strike < strike_prices[0] or strike > strike_prices[-1]:
        continue
    option_price = option_price_interp[i]
    implied_vol = implied_vol_interp[i]
    if i == ATM_index:
        weight = 1.0
    else:
        weight = (strike / spot_price) ** 2 * np.exp(0.25 * implied_vol ** 2 * maturities[ATM_index] - 0.25 * implied_vol ** 2 * maturities[i])
    variance += weight * ((option_price - option_price_interp[ATM_index]) / spot_price) ** 2 / maturities[i]

# Calculate the VIX
VIX = 100 * np.sqrt(variance * 365)
print("VIX for {} on {}: {:.2f}".format(symbol, expiration_date, VIX))

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

# Define the symbol for the S&P500 index
symbol = "SPY"

# Define the expiration date for the options (one month from today)
expiration_date = pd.Timestamp.today() + pd.DateOffset(months=1)

# Get the options data for the symbol and expiration date from Yahoo Finance
options = yf.Ticker(symbol).option_chain(expiration_date.strftime("%Y-%m-%d"))

# Select the OTM and ITM options for the VIX calculation
strike_prices = options.calls['strike']
moneyness = np.log(strike_prices / options.underlying_price)
ATM_index = np.argmin(np.abs(moneyness))
range_indices = np.arange(len(strike_prices))
OTM_indices = range_indices[range_indices < ATM_index]
ITM_indices = range_indices[range_indices > ATM_index]
selected_indices = np.concatenate((OTM_indices[-1::-1], [ATM_index], ITM_indices))

# Calculate the maturities of the options
maturities = np.zeros(len(strike_prices))
for i in range(len(strike_prices)):
    maturities[i] = (expiration_date - pd.to_datetime(options.calls['lastTradeDate'][i])).days / 365.0

# Calculate the option prices and implied volatilities
option_prices = options.calls['lastPrice'][selected_indices]
option_strikes = strike_prices[selected_indices]
option_maturities = maturities[selected_indices]
option_rates = 0.02 * np.ones_like(option_maturities)  # assume a constant interest rate
option_dividends = 0.0 * np.ones_like(option_maturities)  # assume no dividends
option_implied_vols = np.zeros_like(option_maturities)
for i in range(len(option_implied_vols)):
    option_implied_vols[i] = bisect_implied_vol(option_prices[i], options.underlying_price, option_strikes[i],
                                                 option_maturities[i], option_rates[i], option_dividends[i])

# Interpolate the implied volatilities to a range of standardized strike prices
interpolation_kind = 'linear'
std_strikes = np.linspace(-2, 2, num=50)
std_maturities = np.zeros_like(std_strikes)
std_implied_vols = np.zeros_like(std_strikes)
for i in range(len(std_strikes)):
    std_maturities[i] = (30.0 - 1.0/365.0) / 365.0
    std_implied_vols[i] = interp1d(moneyness[selected_indices], option_implied_vols, kind=interpolation_kind)(std_strikes[i])

# Calculate the VIX index
T = std_maturities[0] * 365.0
K = np.exp(std_strikes) * options.underlying_price
option_prices_interp = interp1d(K, option_prices[selected_indices], kind=interpolation_kind)
option_implied_vol_interp = interp1d(K, option_implied_vols[selected_indices], kind=interpolation_kind)

#Define the function to calculate the integrand in the VIX formula
def integrand(k):
    option_price = option_prices_interp(k)[0]
    implied_vol = option_implied_vol_interp(k)[0]
    dK = std_strikes[1] - std_strikes[0]
    return ((np.exp(-rf*T)*dK)/(k**2)) * option_price * (np.exp((rf-q)*T)*norm.cdf(d1(k, options.underlying_price, T, implied_vol)) - norm.cdf(d2(k, options.underlying_price, T, implied_vol)))

#Calculate the VIX index using Simpson's rule
vix = 100 * np.sqrt((2/T)*sum([integrand(K[i]) for i in range(len(K))]))

print(f"VIX for {symbol} on {expiration_date}: {vix:.2f}")